**Load the products silver**

In [0]:
company_a_products_df = spark.table(
    "workspace.silver.company_a_products"
)

company_b_products_df = spark.table(
    "workspace.silver.company_b_products"
)

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    regexp_replace,
    trim,
)

**Company A**

In [0]:
company_a_normalized_df = (
    company_a_products_df
    .withColumn(
        "normalized_product_name",
        lower(
            regexp_replace(
                trim(col("product_name")),
                r"[^a-zA-Z0-9]",
                ""
            )
        )
    )
)

**Company B**

In [0]:
company_b_normalized_df = (
    company_b_products_df
    .withColumn(
        "normalized_product_name",
        lower(
            regexp_replace(
                trim(col("product_name")),
                r"[^a-zA-Z0-9]",
                ""
            )
        )
    )
)

**Matching the normalized names**

In [0]:
exact_matches_df = (
    company_b_normalized_df.alias("b")
    .join(
        company_a_normalized_df.alias("a"),
        col("b.normalized_product_name")
        == col("a.normalized_product_name"),
        "inner"
    )
    .select(
        col("b.product_id").alias("company_b_product_id"),
        col("b.product_name").alias("company_b_product_name"),
        col("a.product_id").alias("company_a_product_id"),
        col("a.product_name").alias("company_a_product_name"),
        col("b.normalized_product_name"),
    )
)

In [0]:
print(
    "Normalized-name matches:",
    exact_matches_df.count()
)

In [0]:
exact_matches_df.show(
    50,
    truncate=False
)

In [0]:
print(
    "Match rows:",
    exact_matches_df.count()
)

print(
    "Distinct Company B products matched:",
    exact_matches_df
    .select("company_b_product_id")
    .distinct()
    .count()
)

In [0]:
(
    exact_matches_df
    .groupBy("company_b_product_id")
    .count()
    .filter(col("count") > 1)
    .show(truncate=False)
)

In [0]:
ambiguous_b_ids_df = (
    exact_matches_df
    .groupBy("company_b_product_id")
    .count()
    .filter(col("count") > 1)
    .select("company_b_product_id")
)

(
    exact_matches_df.alias("m")
    .join(
        ambiguous_b_ids_df.alias("a"),
        col("m.company_b_product_id")
        == col("a.company_b_product_id"),
        "inner"
    )
    .select(
        "m.company_b_product_id",
        "m.company_b_product_name",
        "m.company_a_product_id",
        "m.company_a_product_name",
        "m.normalized_product_name",
    )
    .orderBy(
        "m.company_b_product_id",
        "m.company_a_product_id",
    )
    .show(truncate=False)
)

In [0]:
from pyspark.sql.functions import abs, when, lit

ambiguous_candidates_df = (
    company_b_normalized_df.alias("b")
    .join(
        company_a_normalized_df.alias("a"),
        col("b.normalized_product_name")
        == col("a.normalized_product_name"),
        "inner"
    )
    .join(
        ambiguous_b_ids_df.alias("x"),
        col("b.product_id")
        == col("x.company_b_product_id"),
        "inner"
    )
    .select(
        col("b.product_id").alias("company_b_product_id"),
        col("b.product_name").alias("company_b_product_name"),
        col("b.category").alias("company_b_category"),
        col("b.unit_price").alias("company_b_price"),

        col("a.product_id").alias("company_a_product_id"),
        col("a.product_name").alias("company_a_product_name"),
        col("a.category").alias("company_a_category"),
        col("a.unit_price").alias("company_a_price"),
    )
    .withColumn(
        "category_match",
        when(
            lower(col("company_b_category"))
            == lower(col("company_a_category")),
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "price_difference",
        abs(
            col("company_b_price")
            - col("company_a_price")
        )
    )
)

ambiguous_candidates_df.show(
    truncate=False
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

candidate_window = (
    Window
    .partitionBy("company_b_product_id")
    .orderBy(
        col("category_match").desc(),
        col("price_difference").asc()
    )
)

ranked_ambiguous_df = (
    ambiguous_candidates_df
    .withColumn(
        "candidate_rank",
        row_number().over(candidate_window)
    )
)

In [0]:
ranked_ambiguous_df.select(
    "company_b_product_id",
    "company_b_product_name",
    "company_b_price",
    "company_a_product_id",
    "company_a_product_name",
    "company_a_price",
    "category_match",
    "price_difference",
    "candidate_rank"
).orderBy(
    "company_b_product_id",
    "candidate_rank"
).show(truncate=False)

In [0]:
resolved_ambiguous_matches_df = (
    ranked_ambiguous_df
    .filter(col("candidate_rank") == 1)
)

In [0]:
unambiguous_matches_df = (
    exact_matches_df.alias("m")
    .join(
        ambiguous_b_ids_df.alias("a"),
        col("m.company_b_product_id")
        == col("a.company_b_product_id"),
        "left_anti"
    )
)

In [0]:
print(
    "Unambiguous matches:",
    unambiguous_matches_df.count()
)

In [0]:
resolved_ambiguous_final_df = (
    resolved_ambiguous_matches_df
    .select(
        col("company_b_product_id"),
        col("company_b_product_name"),
        col("company_a_product_id"),
        col("company_a_product_name"),
    )
)

In [0]:
unambiguous_final_df = (
    unambiguous_matches_df
    .select(
        col("company_b_product_id"),
        col("company_b_product_name"),
        col("company_a_product_id"),
        col("company_a_product_name"),
    )
)

In [0]:
matched_product_crosswalk_df = (
    unambiguous_final_df
    .unionByName(
        resolved_ambiguous_final_df
    )
)

In [0]:
print(
    "Final matched rows:",
    matched_product_crosswalk_df.count()
)

print(
    "Distinct Company B products:",
    matched_product_crosswalk_df
    .select("company_b_product_id")
    .distinct()
    .count()
)

In [0]:
(
    matched_product_crosswalk_df
    .groupBy("company_b_product_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
matched_b_ids_df = (
    matched_product_crosswalk_df
    .select(
        col("company_b_product_id")
        .alias("product_id")
    )
)

new_company_b_products_df = (
    company_b_products_df.alias("b")
    .join(
        matched_b_ids_df.alias("m"),
        col("b.product_id") == col("m.product_id"),
        "left_anti"
    )
)

In [0]:
print(
    "New Company B products:",
    new_company_b_products_df.count()
)

In [0]:
from pyspark.sql.functions import lit

matched_product_crosswalk_df = (
    matched_product_crosswalk_df
    .withColumn(
        "match_type",
        lit("MATCHED_EXISTING_PRODUCT")
    )
)

In [0]:
new_product_crosswalk_df = (
    new_company_b_products_df
    .select(
        col("product_id").alias("company_b_product_id"),
        col("product_name").alias("company_b_product_name"),
    )
    .withColumn(
        "company_a_product_id",
        lit(None).cast("string")
    )
    .withColumn(
        "company_a_product_name",
        lit(None).cast("string")
    )
    .withColumn(
        "match_type",
        lit("NEW_PRODUCT")
    )
)

In [0]:
final_product_crosswalk_df = (
    matched_product_crosswalk_df
    .select(
        "company_b_product_id",
        "company_b_product_name",
        "company_a_product_id",
        "company_a_product_name",
        "match_type",
    )
    .unionByName(
        new_product_crosswalk_df
    )
)

In [0]:
print(
    "Crosswalk rows:",
    final_product_crosswalk_df.count()
)

final_product_crosswalk_df \
    .groupBy("match_type") \
    .count() \
    .show()

In [0]:
(
    final_product_crosswalk_df
    .groupBy("company_b_product_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
(
    final_product_crosswalk_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.product_crosswalk"
    )
)

In [0]:
from pyspark.sql.functions import concat, lit, col

new_b_products_canonical_df = (
    new_company_b_products_df
    .select(
        col("product_id").alias("legacy_company_b_product_id"),
        col("product_name"),
        col("category"),
        col("unit_price"),
    )
    .withColumn(
        "canonical_product_id",
        concat(
            lit("BNEW_"),
            col("legacy_company_b_product_id")
        )
    )
)

In [0]:
company_a_master_df = (
    company_a_products_df
    .select(
        col("product_id").alias("canonical_product_id"),
        col("product_name"),
        col("category"),
        col("unit_price"),
    )
    .withColumn(
        "source_system",
        lit("company_a")
    )
)

In [0]:
company_b_new_master_df = (
    new_b_products_canonical_df
    .select(
        "canonical_product_id",
        "product_name",
        "category",
        "unit_price",
    )
    .withColumn(
        "source_system",
        lit("company_b")
    )
)

In [0]:
unified_product_master_df = (
    company_a_master_df
    .unionByName(
        company_b_new_master_df
    )
)

In [0]:
print(
    "Unified products:",
    unified_product_master_df.count()
)

In [0]:
(
    unified_product_master_df
    .groupBy("canonical_product_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
(
    unified_product_master_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.unified_products"
    )
)

In [0]:
product_crosswalk_df = spark.table(
    "workspace.silver.product_crosswalk"
)

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    concat,
    lit,
)

product_crosswalk_canonical_df = (
    product_crosswalk_df
    .withColumn(
        "canonical_product_id",
        when(
            col("match_type") == "MATCHED_EXISTING_PRODUCT",
            col("company_a_product_id")
        )
        .otherwise(
            concat(
                lit("BNEW_"),
                col("company_b_product_id")
            )
        )
    )
)

In [0]:
product_crosswalk_canonical_df.select(
    "company_b_product_id",
    "company_a_product_id",
    "canonical_product_id",
    "match_type"
).show(100, truncate=False)

In [0]:
print(
    "Crosswalk rows:",
    product_crosswalk_canonical_df.count()
)

print(
    "Missing canonical IDs:",
    product_crosswalk_canonical_df
    .filter(
        col("canonical_product_id").isNull()
    )
    .count()
)

In [0]:
(
    product_crosswalk_canonical_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.silver.product_crosswalk"
    )
)

In [0]:
spark.table(
    "workspace.silver.product_crosswalk"
).printSchema()

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(canonical_product_id) AS canonical_ids
FROM workspace.silver.product_crosswalk
""").show()

In [0]:
company_b_order_items_df = spark.table(
    "workspace.silver.company_b_order_items"
)

product_crosswalk_df = spark.table(
    "workspace.silver.product_crosswalk"
)

In [0]:
company_b_order_items_canonical_df = (
    company_b_order_items_df.alias("oi")
    .join(
        product_crosswalk_df.alias("x"),
        col("oi.product_id")
        == col("x.company_b_product_id"),
        "left"
    )
)

In [0]:
company_b_order_items_canonical_df = (
    company_b_order_items_canonical_df
    .select(
        col("oi.order_item_id"),
        col("oi.order_id"),

        col("oi.product_id")
        .alias("legacy_product_id"),

        col("x.canonical_product_id")
        .alias("product_id"),

        col("oi.quantity"),
        col("oi.unit_price"),
        col("oi.discount_pct"),

        col("oi._source_system"),
        col("oi._source_file"),
        col("oi._source_path"),
        col("oi._file_modification_time"),
        col("oi._ingested_at"),
    )
)

In [0]:
print(
    "Canonical order items:",
    company_b_order_items_canonical_df.count()
)

print(
    "Missing canonical product IDs:",
    company_b_order_items_canonical_df
    .filter(col("product_id").isNull())
    .count()
)

In [0]:
unified_products_df = spark.table(
    "workspace.silver.unified_products"
)

invalid_product_refs_df = (
    company_b_order_items_canonical_df.alias("oi")
    .join(
        unified_products_df
        .select("canonical_product_id")
        .alias("p"),
        col("oi.product_id")
        == col("p.canonical_product_id"),
        "left_anti"
    )
)

print(
    "Invalid unified product references:",
    invalid_product_refs_df.count()
)

In [0]:
(
    company_b_order_items_canonical_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_b_order_items_canonical"
    )
)